In [ ]:
!pip install -q Unidecode
!pip install -q piper-tts
!pip install -q pydub
!pip install -q "piper-tts[alignment]"

!git clone https://huggingface.co/MahtaFetrat/Homo-GE2PE-Persian

!unzip -o Homo-GE2PE-Persian/model-weights/homo-ge2pe.zip -d /content/homo-ge2pe
!unzip -o Homo-GE2PE-Persian/model-weights/homo-t5.zip -d /content/homo-t5
!unzip -o Homo-GE2PE-Persian/assets/Parsivar.zip -d /content/
!cp Homo-GE2PE-Persian/assets/GE2PE.py /content/

import os
import shutil

if os.path.exists("/content/g2p_resources") and not os.path.exists("/content/Parsivar/g2p_resources"):
    shutil.move("/content/g2p_resources", "/content/Parsivar/g2p_resources")

os.chdir("/content")

with open("/content/Parsivar/normalizer.py", "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace("./Parsivar/resource/", "/content/Parsivar/resource/")
code = code.replace("./g2p_resources/", "/content/Parsivar/g2p_resources/")

with open("/content/Parsivar/normalizer.py", "w", encoding="utf-8") as f:
    f.write(code)

# رفع ناسازگاری Python 3.12
!sed -i 's/from collections import Iterable/from collections.abc import Iterable/g' /content/Parsivar/token_merger.py
!sed -i 's/from collections import Iterable/from collections.abc import Iterable/g' /content/Parsivar/Parsivar-main/token_merger.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 67.7 MB/s eta 0:00:00
Cloning into 'Homo-GE2PE-Persian'...
remote: Enumerating objects: 58, done.
remote: Total 58 (delta 0), reused 0 (delta 0), pack-reused 58 (from 1)
Receiving objects: 100% (58/58), 42.49 KiB | 10.62 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Filtering content: 100% (3/3), 218.23 MiB | 31.90 MiB/s, done.
Archive:  Homo-GE2PE-Persian/model-weights/homo-ge2pe.zip
  inflating: /content/homo-ge2pe/added_tokens.json  
  inflating: /content/homo-ge2pe/config.json  
  inflating: /content/homo-ge2pe/generation_config.json  
  inflating: /content/homo-ge2pe/model.safetensors  
  inflating: /content/homo-ge2pe/optimizer.pt  
  inflating: /content/homo-ge2pe/rng_state.pth  
  in

In [ ]:
from GE2PE import GE2PE

g2pe_model = GE2PE(model_path="/content/homo-ge2pe")

/content/Parsivar/data_helper.py:29: SyntaxWarning: invalid escape sequence '\s'
  pattern = '\s*' + punctuations + '+' + '\s*'
/content/Parsivar/data_helper.py:29: SyntaxWarning: invalid escape sequence '\s'
  pattern = '\s*' + punctuations + '+' + '\s*'


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
import os
import subprocess

LOCAL_MODELS_PATH = "/content/local_models"

if not os.path.exists(LOCAL_MODELS_PATH):
    os.makedirs(LOCAL_MODELS_PATH)

os.chdir(LOCAL_MODELS_PATH)
print(f"کار در مسیر محلی: {os.getcwd()}")

# ۱. تعریف فایل‌ها و لینک‌های مدل مردانه (گنجی)
ganji_files = {
    "fa_IR-ganji-medium.onnx": "https://huggingface.co/rhasspy/piper-voices/resolve/main/fa/fa_IR/ganji/medium/fa_IR-ganji-medium.onnx",
    "fa_IR-ganji-medium.onnx.json": "https://huggingface.co/rhasspy/piper-voices/resolve/main/fa/fa_IR/ganji/medium/fa_IR-ganji-medium.onnx.json"
}

print("--- بررسی و دانلود مدل گنجی ---")
for fname, furl in ganji_files.items():
    if os.path.exists(fname):
        print(f"فایل {fname} از قبل موجود است.")
    else:
        print(f"در حال دانلود {fname}...")
        subprocess.run(["wget", furl], check=True)


# ۲.تعریف فایل‌ها و لینک‌های مدل زنانه (مانا)
mana_files = {
    "fa_IR-mana-medium.onnx": "https://huggingface.co/MahtaFetrat/Mana-Persian-Piper/resolve/main/fa_IR-mana-medium.onnx",
    "fa_IR-mana-medium.onnx.json": "https://huggingface.co/MahtaFetrat/Mana-Persian-Piper/resolve/main/fa_IR-mana-medium.onnx.json"
}

print("\n--- بررسی و دانلود مدل مانا ---")
for fname, furl in mana_files.items():
    if os.path.exists(fname):
        print(f"فایل {fname} از قبل موجود است.")
    else:
        print(f"در حال دانلود {fname}...")
        subprocess.run(["wget", furl], check=True)


# ۳. تعریف متغیرهای نهایی مسیر مدل‌ها برای استفاده در برنامه‌ی اصلی
female_path = os.path.join(LOCAL_MODELS_PATH, "fa_IR-mana-medium.onnx")
male_path = os.path.join(LOCAL_MODELS_PATH, "fa_IR-ganji-medium.onnx")

print(f"مسیر مدل زنانه: {female_path}")
print(f"مسیر مدل مردانه: {male_path}")

کار در مسیر محلی: /content/local_models
--- بررسی و دانلود مدل گنجی ---
در حال دانلود fa_IR-ganji-medium.onnx...
در حال دانلود fa_IR-ganji-medium.onnx.json...

--- بررسی و دانلود مدل مانا ---
در حال دانلود fa_IR-mana-medium.onnx...
در حال دانلود fa_IR-mana-medium.onnx.json...
مسیر مدل زنانه: /content/local_models/fa_IR-mana-medium.onnx
مسیر مدل مردانه: /content/local_models/fa_IR-ganji-medium.onnx


In [ ]:
from piper import PiperVoice, SynthesisConfig
from pydub import AudioSegment
import re
import os
import uuid
import glob
import time
import gradio as gr


# =========================================================
# فونم‌هایی که از دو یا چند کاراکتر تشکیل شده‌اند
# =========================================================
MULTI_PHONEMES = ("tʃ", "dʒ", "qq")
# =========================================================
# فرهنگ نگاشت (PHONEME_MAPPING)برای تمام کاراکترهای واجی
# =========================================================
PHONEME_MAPPING = {
    '/': 'a',      # فتحه / مصوت کوتاه آ
    'a': 'ɑ',      # آ بلند
    'e': 'e',      # کسره
    'o': 'o',      # ضمه
    'i': 'i',      # ای
    'u': 'u',      # او
    '@': 'ʔ',      # همزه
    '?': 'ʔ',      # همزه جایگزین
    '$': 'ʃ',      # ش
    'c': 'tʃ',     # چ
    ';': 'ʒ',      # ژ
    'g': 'ɡ',      # گ
    'y': 'j',      # ی (صامت)
    'q': 'qq',     # ق
    'ey': 'ej',    # ای
    '1': '',
}
PUNCTUATION = ",."

# =========================================================
# تبدیل فونم Homo به فونم قابل فهم برای Piper
# =========================================================
def homo_to_exact_mana(homo_phonemic_text, original_text):
    homo_phonemic_text = re.sub(r'(?<!\w)@ali\b', '@/li', homo_phonemic_text)
    homo_phonemic_text = re.sub(r'h\$il', 'hsil', homo_phonemic_text)

    orig_tokens = re.findall(r'\w+|[^\w\s]', original_text, flags=re.UNICODE)
    homo_tokens = homo_phonemic_text.split()

    rebuilt = []
    h = 0

    for tok in orig_tokens:
        if re.fullmatch(r'[،,.;؛:!?؟]', tok):
            if rebuilt:
                if tok in "،,؛;:":
                    rebuilt[-1] += ","
                elif tok in ".!?؟":
                    rebuilt[-1] = rebuilt[-1].replace("1", "")
                    if rebuilt[-1].endswith("e"):
                        rebuilt[-1] = rebuilt[-1][:-1]
                    rebuilt[-1] += "."
            continue

        if h >= len(homo_tokens):
            break

        rebuilt.append(homo_tokens[h])
        h += 1

    homo_phonemic_text = " ".join(rebuilt).replace("1", "")
    translated = []
    has_zhe = "ژ" in original_text

    for ch in homo_phonemic_text:
        if ch in PUNCTUATION:
            translated.append(ch)
            continue
        if ch == 'j':
            translated.append('ʒ' if has_zhe else 'dʒ')
            continue
        translated.append(PHONEME_MAPPING.get(ch, ch))

    return "".join(translated)

# =========================================================
# 🔢 شمارش تعداد فونم‌های موجود در رشته IPA
# =========================================================
def count_phonemes(ipa):
    i = 0
    count = 0
    len_ipa = len(ipa)

    while i < len_ipa:
        matched = False
        for phoneme in MULTI_PHONEMES:
            if ipa.startswith(phoneme, i):
                count += 1
                i += len(phoneme)
                matched = True
                break
        if not matched:
            if ipa[i] != " ":
                count += 1
            i += 1

    return count
# =========================================================
#  تبدیل متن به فونم نهایی Piper
# =========================================================
def text_to_piper_phoneme(text):
    homo = g2pe_model.generate([text], use_rules=True)[0]
    return homo_to_exact_mana(homo, text)

# =========================================================
# بارگذاری Piper فقط یک بار
# =========================================================
female_voice = None
male_voice = None

def get_piper_voice(model_path, is_female):
    global female_voice, male_voice

    if is_female:
        if female_voice is None:
            female_voice = PiperVoice.load(
                model_path,
                include_alignments=True
            )
        return female_voice

    else:
        if male_voice is None:
            male_voice = PiperVoice.load(
                model_path,
                include_alignments=True
            )
        return male_voice
# =========================================================
#  تولید گفتار املایی
# =========================================================
def generate_spelling_speech_fast(
    sentence_phonemes_list,
    output_wave_path,
    model_path,
    speed=0.6,
    pause_per_phoneme=80,
    is_female=False
):
    voice = get_piper_voice(model_path, is_female)

    result = AudioSegment.empty()

    syn_config = SynthesisConfig(
        length_scale=1.0 / speed
    )

    for sentence_idx, full_phoneme in enumerate(sentence_phonemes_list):

        original_words = full_phoneme.split()

        # -------------------------------------------------
        # پیشوند کمکی برای جمله‌های خیلی کوتاه مدل زنانه
        # -------------------------------------------------
        needs_prefix = is_female and len(original_words) <= 2

        if needs_prefix:
            prefix = "kalameh moredenazare"
            piper_input = f"[[ {prefix} {full_phoneme} ]]"
        else:
            piper_input = f"[[ {full_phoneme} ]]"

        chunk = next(
            voice.synthesize(
              piper_input,
              syn_config=syn_config,
              include_alignments=True
            ),
            None
        )

        if chunk is None:
            continue

        sentence_audio = AudioSegment(
            chunk.audio_int16_bytes,
            frame_rate=chunk.sample_rate,
            sample_width=chunk.sample_width,
            channels=chunk.sample_channels
        )

        alignments = chunk.phoneme_alignments

        if not alignments:
            raise RuntimeError(
                "Alignment برای این مدل Piper موجود نیست."
            )

        words = []

        current_word_start = None
        current_word_end = None

        current_time_ms = 0.0

        for alignment in alignments:

            phoneme = alignment.phoneme

            duration_ms = (
                alignment.num_samples /
                chunk.sample_rate
            ) * 1000.0

            start_ms = current_time_ms
            end_ms = current_time_ms + duration_ms

            current_time_ms = end_ms

            # BOS / EOS
            if phoneme in ("^", "$"):
                continue

            # فاصله = پایان کلمه
            if phoneme == " ":

                if current_word_start is not None:
                    words.append(
                        (
                            current_word_start,
                            current_word_end
                        )
                    )

                current_word_start = None
                current_word_end = None

                continue

            # اولین فونم کلمه
            if current_word_start is None:
                current_word_start = start_ms

            current_word_end = end_ms

        # آخرین کلمه
        if current_word_start is not None:
            words.append(
                (
                    current_word_start,
                    current_word_end
                )
            )

        # -------------------------------------------------
        # حذف کلمات prefix
        # -------------------------------------------------
        if needs_prefix:

            prefix_word_count = len(prefix.split())

            if len(words) > prefix_word_count:
                words = words[prefix_word_count:]

            else:
                raise RuntimeError(
                    "کلمات prefix از کلمات اصلی بیشتر یا مساوی شده‌اند."
                )

        # -------------------------------------------------
        # تولید خروجی املایی
        # -------------------------------------------------
        for word_index, (start_ms, end_ms) in enumerate(words):

            # کمی از ابتدا و انتهای کلمه حفظ می‌شود
            # تا برش کاملاً خشک نباشد.
            edge_ms = 15

            start_ms = max(
                0,
                start_ms - edge_ms
            )

            end_ms = min(
                len(sentence_audio),
                end_ms + edge_ms
            )

            word_audio = sentence_audio[
                int(start_ms):int(end_ms)
            ]

            fade_in_ms = 5
            fade_out_ms = 10

            if len(word_audio) > fade_in_ms + fade_out_ms:
                word_audio = (
                    word_audio
                    .fade_in(fade_in_ms)
                    .fade_out(fade_out_ms)
                )

            result += word_audio

            if word_index < len(words) - 1:

                # تعداد فونم کلمه اصلی
                phoneme_count = count_phonemes(original_words[word_index])

                pause_ms = int(
                    phoneme_count * pause_per_phoneme
                )

                result += AudioSegment.silent(
                    duration=pause_ms
                )

        if sentence_idx < len(sentence_phonemes_list) - 1:

            sentence_phoneme_count = sum(
                count_phonemes(word)
                for word in original_words
            )

            sentence_pause_ms = int(
                sentence_phoneme_count *
                pause_per_phoneme
            )

            result += AudioSegment.silent(
                duration=sentence_pause_ms
            )

    # -------------------------------------------------
    # ذخیره خروجی نهایی
    # -------------------------------------------------
    result.export(
        output_wave_path,
        format="wav"
    )

    return output_wave_path
# =========================================================
#  تولید گفتار در حالت عادی
# =========================================================
def generate_normal_speech(
    full_phonemes,
    output_wave_path,
    model_path,
    speed=0.7,
    is_female=True,
    text_input_len=0
):
    voice = get_piper_voice(model_path, is_female)

    needs_prefix = is_female and text_input_len <= 2

    if needs_prefix:
        prefix = "kalameh moredenazare"
        piper_input = f"[[ {prefix} {full_phonemes} ]]"
    else:
        piper_input = f"[[ {full_phonemes} ]]"

    syn_config = SynthesisConfig(
        length_scale=1.0 / speed
    )

    chunks = list(
        voice.synthesize(
            piper_input,
            syn_config=syn_config,
            include_alignments=True
        )
    )

    if not chunks:
        AudioSegment.empty().export(
            output_wave_path,
            format="wav"
        )
        return output_wave_path

    chunk = chunks[0]

    audio = AudioSegment(
        chunk.audio_int16_bytes,
        frame_rate=chunk.sample_rate,
        sample_width=chunk.sample_width,
        channels=chunk.sample_channels
    )

    if not needs_prefix:
        audio.export(
            output_wave_path,
            format="wav"
        )
        return output_wave_path

    alignments = chunk.phoneme_alignments

    if not alignments:
        raise RuntimeError(
            "Alignment برای این مدل Piper موجود نیست."
        )

    prefix_word_count = len(prefix.split())

    current_time_ms = 0.0
    spaces_seen = 0
    cut_time_ms = 0.0

    for alignment in alignments:

        duration_ms = (
            alignment.num_samples /
            chunk.sample_rate
        ) * 1000.0

        current_time_ms += duration_ms

        if alignment.phoneme == " ":
            spaces_seen += 1

            if spaces_seen == prefix_word_count:
                cut_time_ms = current_time_ms
                break

    if cut_time_ms <= 0:
        raise RuntimeError(
            "مرز prefix در alignment پیدا نشد."
        )

    safety_ms = 10

    cut_time_ms = max(
        0,
        cut_time_ms - safety_ms
    )

    audio = audio[int(cut_time_ms):]

    audio.export(
        output_wave_path,
        format="wav"
    )

    return output_wave_path
# ==========================================================
#  رابط کاربری Gradio
# =========================================================
def main_interface(text, reading_mode, gender_selection, pause_per_phoneme):
    model_path = female_path if gender_selection == "زنانه (Female)" else male_path
    is_female = (gender_selection == "زنانه (Female)")

    now = time.time()
    for f in glob.glob("output_*.wav"):
        if now - os.path.getmtime(f) > 300:
            try:
                os.remove(f)
            except OSError:
                pass

    output_path = f"output_{uuid.uuid4().hex}.wav"
    sentences = [s.strip() for s in re.split(r'[.!؟?]+', text.strip()) if s.strip()]

    sentence_phonemes_cache = [text_to_piper_phoneme(s) for s in sentences]

    report_lines = []
    for idx, ipa in enumerate(sentence_phonemes_cache, 1):
        count = sum(count_phonemes(word) for word in ipa.split())
        report_lines.append(f"جمله {idx}: {count} فونم")

    sentence_report = "\n".join(report_lines)

    if reading_mode == "املایی":
      output_path = generate_spelling_speech_fast(
        sentence_phonemes_list=sentence_phonemes_cache,
        output_wave_path=output_path,
        model_path=model_path,
        pause_per_phoneme=pause_per_phoneme,
        is_female=is_female
      )
    else:
        full_text_phonemes = " ".join(sentence_phonemes_cache)
        output_path = generate_normal_speech(
            full_phonemes=full_text_phonemes,
            output_wave_path=output_path,
            model_path=model_path,
            is_female=is_female,
            text_input_len=len(text.split())
        )

    final_output_joined = " ".join(sentence_phonemes_cache)
    report = (
        f"متن ورودی : {text}\n"
        f"{final_output_joined} : فونم‌های نهایی\n\n"
        f"تعداد فونم هر جمله:\n{sentence_report}"
    )

    return output_path, report

with gr.Blocks() as demo:
    gr.Markdown("# 🎙️ سیستم تولید گفتار")

    reading_mode = gr.Radio(
        choices=["عادی", "املایی"],
        value="عادی",
        label="حالت خواندن"
    )

    with gr.Row():
        input_text = gr.Textbox(label="متن ورودی")
        gender = gr.Radio(
            choices=["زنانه (Female)", "مردانه (Male)"],
            value="زنانه (Female)",
            label="گوینده"
        )
        pause_slider = gr.Slider(
            minimum=20,
            maximum=2000,
            value=80,
            step=5,
            label="مکث به ازای هر فونم (ms)",
            interactive=False
        )

    def update_ui(mode):
        return gr.update(interactive=(mode == "املایی"))

    reading_mode.change(
        fn=update_ui,
        inputs=reading_mode,
        outputs=pause_slider
    )

    btn = gr.Button("تولید صدا")
    audio_out = gr.Audio(label="خروجی")
    report_out = gr.Textbox(label="گزارش", lines=6)

    btn.click(
        main_interface,
        inputs=[input_text, reading_mode, gender, pause_slider],
        outputs=[audio_out, report_out]
    )

demo.launch(debug=False, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://564d2c83b6c98521f8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
